Extract data from csv

In [319]:
import pandas as pd
import numpy as np

In [320]:
df = pd.read_csv('data/bronze/dirty_cafe_sales.csv')

In [321]:
df.sample(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9148,TXN_4127927,Salad,5,5.0,25.0,NaN,Takeaway,2023-08-04
4826,TXN_7475500,Cookie,2,1.0,2.0,Cash,NaN,2023-11-22
5939,TXN_7986904,Cookie,3,1.0,3.0,Credit Card,Takeaway,2023-08-11
5639,TXN_6206792,Tea,NaN,ERROR,6.0,Credit Card,NaN,2023-10-13
9074,TXN_2119809,Salad,5,5.0,25.0,Digital Wallet,In-store,2023-04-21


In [322]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [323]:
df['Item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

In [324]:
df_copy = df.copy()

In [325]:
df_copy.columns = [c.lower().replace(' ', '_') for c in df_copy.columns]

In [326]:
df_copy.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [327]:
df_copy.replace(['ERROR', 'UNKNOWN', 'nan'], np.nan, inplace=True)

In [328]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9031 non-null   object
 2   quantity          9521 non-null   object
 3   price_per_unit    9467 non-null   object
 4   total_spent       9498 non-null   object
 5   payment_method    6822 non-null   object
 6   location          6039 non-null   object
 7   transaction_date  9540 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [329]:
df_copy['price_per_unit'] = df_copy['price_per_unit'].astype(float)

In [330]:
df_copy['total_spent'] = df_copy['total_spent'].astype(float)

In [331]:
df_copy['quantity'] = pd.to_numeric(df_copy['quantity'], errors='coerce').astype('Int64')

In [332]:
type(df_copy['quantity'][0])

numpy.int64

In [333]:
df_copy.loc[
   (df_copy['price_per_unit'].isna() &
    df_copy['total_spent'].notna() &
    df_copy['quantity'].notna()
    ), 'price_per_unit'] = df_copy['total_spent']/df_copy['quantity']

In [334]:
df_copy['quantity'] = df_copy['quantity'].replace('<NA>', np.nan)

In [335]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10000 non-null  object 
 1   item              9031 non-null   object 
 2   quantity          9521 non-null   Int64  
 3   price_per_unit    9962 non-null   float64
 4   total_spent       9498 non-null   float64
 5   payment_method    6822 non-null   object 
 6   location          6039 non-null   object 
 7   transaction_date  9540 non-null   object 
dtypes: Int64(1), float64(2), object(5)
memory usage: 634.9+ KB


In [336]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', nan, 'Sandwich',
       'Juice', 'Tea'], dtype=object)

In [337]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
14,TXN_8915701,NaN,2,1.5,3.0,NaN,In-store,2023-03-21
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02
31,TXN_8927252,NaN,2,1.0,NaN,Credit Card,NaN,2023-11-06
...,...,...,...,...,...,...,...,...
9951,TXN_4122925,NaN,4,1.0,4.0,NaN,Takeaway,2023-10-20
9958,TXN_4125474,NaN,2,5.0,10.0,Credit Card,In-store,2023-08-02
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08


In [338]:
mapping_df = df_copy.loc[(df_copy['item'].notna()) & (df_copy['price_per_unit'].notna()), ['item', 'price_per_unit']].drop_duplicates()

In [339]:
mapping = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [340]:
mapping

{'Coffee': 2.0,
 'Cake': 3.0,
 'Cookie': 1.0,
 'Salad': 5.0,
 'Smoothie': 4.0,
 'Sandwich': 4.0,
 'Juice': 3.0,
 'Tea': 1.5}

In [341]:
for item, price in mapping.items():
   mask = (df_copy['item'].isna()) & (df_copy['price_per_unit'] == price)
   df_copy.loc[mask, 'item'] = item

In [342]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
1761,TXN_3611851,NaN,4,NaN,NaN,Credit Card,NaN,2023-02-09
2289,TXN_7524977,NaN,4,NaN,NaN,NaN,NaN,2023-12-09
3779,TXN_7376255,NaN,<NA>,NaN,25.0,NaN,In-store,2023-05-27
4152,TXN_9646000,NaN,2,NaN,NaN,NaN,In-store,2023-12-14
7597,TXN_1082717,NaN,<NA>,NaN,9.0,Digital Wallet,In-store,2023-12-13
9819,TXN_1208561,NaN,<NA>,NaN,20.0,Credit Card,NaN,2023-08-19


In [343]:
for item, price in mapping.items():
   mask = (df_copy['price_per_unit'].isna()) & (df_copy['item'] == item)
   df_copy.loc[mask, 'price_per_unit'] = price

In [344]:
df_copy.loc[
   (df_copy['price_per_unit'].notna() &
    df_copy['total_spent'].isna() &
    df_copy['quantity'].notna()
    ), 'total_spent'] = df_copy['price_per_unit'] * df_copy['quantity']

In [345]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10000 non-null  object 
 1   item              9994 non-null   object 
 2   quantity          9521 non-null   Int64  
 3   price_per_unit    9994 non-null   float64
 4   total_spent       9977 non-null   float64
 5   payment_method    6822 non-null   object 
 6   location          6039 non-null   object 
 7   transaction_date  9540 non-null   object 
dtypes: Int64(1), float64(2), object(5)
memory usage: 634.9+ KB


In [346]:
df_copy[df_copy['total_spent'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
236,TXN_8562645,Salad,<NA>,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,<NA>,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,<NA>,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,<NA>,4.0,NaN,NaN,Takeaway,2023-05-14
1761,TXN_3611851,NaN,4,NaN,NaN,Credit Card,NaN,2023-02-09
2289,TXN_7524977,NaN,4,NaN,NaN,NaN,NaN,2023-12-09
2796,TXN_9188692,Cake,<NA>,3.0,NaN,Credit Card,NaN,2023-12-01
3203,TXN_4565754,Smoothie,<NA>,4.0,NaN,Digital Wallet,Takeaway,2023-10-06
3224,TXN_6297232,Coffee,<NA>,2.0,NaN,NaN,NaN,2023-04-07
3401,TXN_3251829,Tea,<NA>,1.5,NaN,Digital Wallet,In-store,2023-07-25
